# Снижение стоимости малого GPT с помощью локального причинного внимания

Этот Colab-ноутбук сравнивает две decoder-only языковые модели примерно по 49 млн параметров.

- **Baseline:** полное causal multi-head self-attention.
- **Proposed method:** causal sliding-window multi-head self-attention с окном `W=128`.

Во всем остальном модели совпадают: 12 блоков, `d_model=512`, 8 голов, RoPE, RMSNorm в схеме Pre-LN, SwiGLU, residual connections, weight tying, AdamW и cosine schedule с warmup. Ноутбук обучает обе модели на WikiText-103 с одним SentencePiece-токенизатором, измеряет качество, скорость, VRAM, KV-cache, FLOPs и перенос на LAMBADA без дообучения.

**Порядок работы:** включите GPU в `Runtime -> Change runtime type`, выполните ячейки сверху вниз, сначала используйте режим `smoke`, затем `quick`, и только после проверки запускайте `full`.


In [ ]:
# Установка только тех пакетов, которых обычно нет в стандартном образе Colab.
!pip -q install "datasets>=3.0,<5" "sentencepiece>=0.2,<0.3" "pandas>=2.0" "matplotlib>=3.7"


## Конфигурация и воспроизводимость

Режимы различаются числом документов, шагов и объемом оценки. Архитектура во всех режимах остается той же. `smoke` проверяет, что весь код исполняется; его метрики нельзя использовать как научный результат. `quick` подходит для пилота. `full` предназначен для итогового сравнения в рамках одной длительной Colab-сессии.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import platform
import random
import time
from array import array
from contextlib import nullcontext
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset

torch.set_float32_matmul_precision("high")

RUN_MODE = "quick"       # @param ["smoke", "quick", "full"]
USE_GOOGLE_DRIVE = True   # @param {type:"boolean"}
RETRAIN_EXISTING = False  # @param {type:"boolean"}
SEED = 42                 # @param {type:"integer"}

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive/efficient_transformer_research")
    except ImportError:
        PROJECT_DIR = Path.cwd() / "efficient_transformer_research"
else:
    PROJECT_DIR = Path.cwd() / "efficient_transformer_research"

for subdir in ["tokenizer", "data_cache", "checkpoints", "results", "figures"]:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)
print("Project directory:", PROJECT_DIR)


In [ ]:
@dataclass
class RunConfig:
    dataset_name: str = "Salesforce/wikitext"
    dataset_config: str = "wikitext-103-raw-v1"
    vocab_size: int = 16_000
    seq_len: int = 512
    window_size: int = 128
    d_model: int = 512
    n_heads: int = 8
    n_layers: int = 12
    d_ff: int = 1536
    dropout: float = 0.0
    learning_rate: float = 3e-4
    min_lr_ratio: float = 0.1
    weight_decay: float = 0.1
    betas: tuple[float, float] = (0.9, 0.95)
    grad_clip: float = 1.0
    warmup_ratio: float = 0.05
    batch_size: int = 2
    grad_accum_steps: int = 8
    num_workers: int = 2
    train_steps: int = 300
    eval_batches: int = 40
    log_every: int = 20
    max_train_docs: int | None = 250_000
    max_train_tokens: int | None = 20_000_000
    max_validation_tokens: int = 2_000_000
    tokenizer_train_lines: int = 250_000
    lambada_examples: int = 200

PRESETS = {
    "smoke": dict(train_steps=20, eval_batches=5, log_every=5,
                  max_train_docs=10_000, max_train_tokens=1_000_000,
                  max_validation_tokens=100_000, tokenizer_train_lines=10_000,
                  grad_accum_steps=1, lambada_examples=30),
    "quick": dict(),
    "full": dict(train_steps=2_000, eval_batches=200, log_every=50,
                 max_train_docs=None, max_train_tokens=100_000_000,
                 max_validation_tokens=5_000_000, tokenizer_train_lines=500_000,
                 lambada_examples=1_000),
}

cfg = RunConfig(**{**asdict(RunConfig()), **PRESETS[RUN_MODE]})
assert cfg.d_model % cfg.n_heads == 0
assert 0 < cfg.window_size <= cfg.seq_len
TOKENS_PER_OPTIMIZER_STEP = cfg.batch_size * cfg.grad_accum_steps * cfg.seq_len
print(json.dumps(asdict(cfg), ensure_ascii=False, indent=2))
print("Tokens per optimizer step:", f"{TOKENS_PER_OPTIMIZER_STEP:,}")
print("Planned training tokens per model:", f"{TOKENS_PER_OPTIMIZER_STEP * cfg.train_steps:,}")


## Данные и SentencePiece

WikiText-103 содержит статьи Wikipedia и подходит для языкового моделирования с более длинными зависимостями. Токенизатор BPE обучается только на train split. Специальный токен `eos` разделяет документы. Validation split нигде не участвует в обновлении весов.

Токены сохраняются в Google Drive, поэтому повторный запуск пропускает самую медленную часть подготовки данных.


In [ ]:
raw = load_dataset(cfg.dataset_name, cfg.dataset_config)
train_split = raw["train"]
valid_split = raw["validation"]
print(raw)

sp_prefix = PROJECT_DIR / "tokenizer" / f"wikitext_bpe_{cfg.vocab_size}_lines{cfg.tokenizer_train_lines}"
sp_model_path = Path(str(sp_prefix) + ".model")
sp_corpus_path = PROJECT_DIR / "tokenizer" / f"sentencepiece_corpus_{cfg.tokenizer_train_lines}.txt"

if not sp_model_path.exists():
    with sp_corpus_path.open("w", encoding="utf-8") as f:
        written = 0
        for row in train_split:
            text = row["text"].strip()
            if not text:
                continue
            f.write(text.replace("\n", " ") + "\n")
            written += 1
            if written >= cfg.tokenizer_train_lines:
                break
    spm.SentencePieceTrainer.train(
        input=str(sp_corpus_path),
        model_prefix=str(sp_prefix),
        vocab_size=cfg.vocab_size,
        model_type="bpe",
        character_coverage=0.9995,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        byte_fallback=True,
        hard_vocab_limit=False,
        num_threads=os.cpu_count() or 2,
    )

sp = spm.SentencePieceProcessor(model_file=str(sp_model_path))
assert sp.vocab_size() == cfg.vocab_size, (sp.vocab_size(), cfg.vocab_size)
print("Tokenizer vocabulary:", sp.vocab_size())
print(sp.encode("A small Transformer is ready.", out_type=str))


In [ ]:
def encode_split(split, token_limit: int | None, doc_limit: int | None = None) -> torch.Tensor:
    # vocab < 65536, поэтому временно храним ids как uint16 и экономим RAM.
    ids = array("H")
    docs = 0
    for row in split:
        text = row["text"].strip()
        if not text:
            continue
        encoded = sp.encode(text, out_type=int)
        remaining = None if token_limit is None else token_limit - len(ids)
        if remaining is not None and remaining <= 1:
            break
        if remaining is not None and len(encoded) + 1 > remaining:
            encoded = encoded[: max(0, remaining - 1)]
        ids.extend(encoded)
        ids.append(sp.eos_id())
        docs += 1
        if (doc_limit is not None and docs >= doc_limit) or (token_limit is not None and len(ids) >= token_limit):
            break
    tensor = torch.tensor(ids, dtype=torch.int32)
    print(f"Encoded {docs:,} documents and {tensor.numel():,} tokens")
    return tensor

cache_tag = f"{RUN_MODE}_v{cfg.vocab_size}"
train_cache = PROJECT_DIR / "data_cache" / f"train_{cache_tag}.pt"
valid_cache = PROJECT_DIR / "data_cache" / f"validation_{cache_tag}.pt"

if train_cache.exists():
    train_tokens = torch.load(train_cache, map_location="cpu", weights_only=True)
else:
    train_tokens = encode_split(train_split, cfg.max_train_tokens, cfg.max_train_docs)
    torch.save(train_tokens, train_cache)

if valid_cache.exists():
    valid_tokens = torch.load(valid_cache, map_location="cpu", weights_only=True)
else:
    valid_tokens = encode_split(valid_split, cfg.max_validation_tokens)
    torch.save(valid_tokens, valid_cache)

print("Train tokens:", f"{train_tokens.numel():,}")
print("Validation tokens:", f"{valid_tokens.numel():,}")


In [ ]:
class TokenBlocks(Dataset):
    def __init__(self, tokens: torch.Tensor, seq_len: int):
        self.tokens = tokens
        self.seq_len = seq_len
        self.n_blocks = max(0, (tokens.numel() - 1) // seq_len)

    def __len__(self):
        return self.n_blocks

    def __getitem__(self, index):
        start = index * self.seq_len
        chunk = self.tokens[start : start + self.seq_len + 1].long()
        return chunk[:-1], chunk[1:]

train_dataset = TokenBlocks(train_tokens, cfg.seq_len)
valid_dataset = TokenBlocks(valid_tokens, cfg.seq_len)
assert len(train_dataset) > 0 and len(valid_dataset) > 0
print("Train blocks:", len(train_dataset), "Validation blocks:", len(valid_dataset))

def make_loaders(seed: int):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=cfg.num_workers,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=cfg.num_workers > 0,
        generator=generator,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=cfg.num_workers,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=cfg.num_workers > 0,
    )
    return train_loader, valid_loader


## Архитектура модели

В baseline каждая позиция видит все предыдущие позиции. В local-варианте каждая голова видит только текущий токен и последние `W-1` токенов. Реализация локального пути формирует представление окон через `unfold` и материализует матрицу размера `T x W`, а не `T x T`. Это существенно: обычная плотная маска со значениями `-inf` изменила бы связи, но не гарантировала бы снижения вычислений.


In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 16_000
    d_model: int = 512
    n_heads: int = 8
    n_layers: int = 12
    d_ff: int = 1536
    max_seq_len: int = 512
    dropout: float = 0.0
    attention_type: str = "dense"
    window_size: int = 128
    rope_theta: float = 10_000.0
    rms_eps: float = 1e-6


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        original_dtype = x.dtype
        x_float = x.float()
        x_norm = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_norm * self.weight.float()).to(original_dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, theta: float = 10_000.0):
        super().__init__()
        assert dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, positions: torch.Tensor, dtype: torch.dtype):
        freqs = torch.outer(positions.float(), self.inv_freq.float())
        cos = freqs.cos().repeat_interleave(2, dim=-1)[None, None, :, :]
        sin = freqs.sin().repeat_interleave(2, dim=-1)[None, None, :, :]
        return cos.to(dtype), sin.to(dtype)


def rotate_half(x):
    x_pair = x.float().reshape(*x.shape[:-1], -1, 2)
    x_rot = torch.stack((-x_pair[..., 1], x_pair[..., 0]), dim=-1).flatten(-2)
    return x_rot.to(x.dtype)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.attention_type = config.attention_type
        self.window_size = config.window_size
        self.dropout = config.dropout
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model, bias=False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, config.rope_theta)

    def _local_attention(self, q, k, v):
        # q, k, v: [B, H, T, Dh]. unfold возвращает view окон, scores имеют [B,H,T,W].
        _, _, t, _ = q.shape
        w = min(self.window_size, t)
        k_pad = F.pad(k, (0, 0, w - 1, 0))
        v_pad = F.pad(v, (0, 0, w - 1, 0))
        k_windows = k_pad.unfold(2, w, 1).permute(0, 1, 2, 4, 3)
        v_windows = v_pad.unfold(2, w, 1).permute(0, 1, 2, 4, 3)
        scores = torch.einsum("bhtd,bhtwd->bhtw", q, k_windows) / math.sqrt(self.head_dim)
        query_pos = torch.arange(t, device=q.device)[:, None]
        offsets = torch.arange(w, device=q.device)[None, :] - (w - 1)
        valid = (query_pos + offsets) >= 0
        scores = scores.masked_fill(~valid[None, None, :, :], torch.finfo(scores.dtype).min)
        probs = F.softmax(scores.float(), dim=-1).to(q.dtype)
        probs = F.dropout(probs, p=self.dropout, training=self.training)
        return torch.einsum("bhtw,bhtwd->bhtd", probs, v_windows)

    def forward(self, x, past_kv=None, use_cache=False, position_offset=0):
        b, t, c = x.shape
        q, k, v = self.qkv(x).split(self.d_model, dim=-1)
        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)

        positions = torch.arange(position_offset, position_offset + t, device=x.device)
        cos, sin = self.rope(positions, q.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is None:
            if self.attention_type == "dense":
                y = F.scaled_dot_product_attention(
                    q, k, v,
                    dropout_p=self.dropout if self.training else 0.0,
                    is_causal=True,
                )
            else:
                y = self._local_attention(q, k, v)
            k_all, v_all = k, v
        else:
            # В ноутбуке cached path применяется для autoregressive decoding по одному токену.
            assert t == 1, "Cached decoding expects one new token per call"
            past_k, past_v = past_kv
            k_all = torch.cat((past_k, k), dim=2)
            v_all = torch.cat((past_v, v), dim=2)
            if self.attention_type == "local":
                k_all = k_all[:, :, -self.window_size :, :]
                v_all = v_all[:, :, -self.window_size :, :]
            y = F.scaled_dot_product_attention(q, k_all, v_all, dropout_p=0.0, is_causal=False)

        new_cache = None
        if use_cache:
            if self.attention_type == "local":
                keep = max(1, self.window_size - 1)
                new_cache = (k_all[:, :, -keep:, :], v_all[:, :, -keep:, :])
            else:
                new_cache = (k_all, v_all)

        y = y.transpose(1, 2).contiguous().view(b, t, c)
        return self.out_proj(y), new_cache


class SwiGLU(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.w_gate = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w_up = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w_down = nn.Linear(config.d_ff, config.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.attn_norm = RMSNorm(config.d_model, config.rms_eps)
        self.ffn_norm = RMSNorm(config.d_model, config.rms_eps)
        self.attn = CausalSelfAttention(config)
        self.ffn = SwiGLU(config)
        self.dropout = config.dropout

    def forward(self, x, past_kv=None, use_cache=False, position_offset=0):
        attn_out, new_cache = self.attn(
            self.attn_norm(x), past_kv=past_kv,
            use_cache=use_cache, position_offset=position_offset,
        )
        x = x + F.dropout(attn_out, p=self.dropout, training=self.training)
        x = x + F.dropout(self.ffn(self.ffn_norm(x)), p=self.dropout, training=self.training)
        return x, new_cache


class SmallGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.final_norm = RMSNorm(config.d_model, config.rms_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, past_key_values=None, use_cache=False, position_offset=0):
        x = self.token_embedding(idx)
        new_past = [] if use_cache else None
        for layer_index, block in enumerate(self.blocks):
            layer_past = None if past_key_values is None else past_key_values[layer_index]
            x, layer_cache = block(
                x, past_kv=layer_past, use_cache=use_cache,
                position_offset=position_offset,
            )
            if use_cache:
                new_past.append(layer_cache)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return {"logits": logits, "loss": loss, "past_key_values": new_past}


def make_gpt_config(attention_type: str) -> GPTConfig:
    return GPTConfig(
        vocab_size=cfg.vocab_size,
        d_model=cfg.d_model,
        n_heads=cfg.n_heads,
        n_layers=cfg.n_layers,
        d_ff=cfg.d_ff,
        max_seq_len=cfg.seq_len,
        dropout=cfg.dropout,
        attention_type=attention_type,
        window_size=cfg.window_size,
    )


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for attention_type in ("dense", "local"):
    test_model = SmallGPT(make_gpt_config(attention_type))
    n_params = count_parameters(test_model)
    print(attention_type, f"{n_params:,} parameters ({n_params / 1e6:.2f}M)")
    assert 45_000_000 <= n_params <= 55_000_000
    assert test_model.lm_head.weight.data_ptr() == test_model.token_embedding.weight.data_ptr()
    del test_model
gc.collect()

# Проверка математической эквивалентности: если окно покрывает всю короткую
# последовательность, local attention должен совпасть с dense causal attention.
tiny_dense_cfg = GPTConfig(vocab_size=128, d_model=32, n_heads=4, n_layers=2,
                           d_ff=96, max_seq_len=16, attention_type="dense",
                           window_size=16, dropout=0.0)
tiny_local_cfg = replace(tiny_dense_cfg, attention_type="local")
seed_everything(123)
tiny_dense = SmallGPT(tiny_dense_cfg).eval()
tiny_local = SmallGPT(tiny_local_cfg).eval()
tiny_local.load_state_dict(tiny_dense.state_dict())
tiny_input = torch.randint(0, tiny_dense_cfg.vocab_size, (2, 16))
with torch.inference_mode():
    dense_logits = tiny_dense(tiny_input)["logits"]
    local_logits = tiny_local(tiny_input)["logits"]
torch.testing.assert_close(dense_logits, local_logits, rtol=1e-4, atol=1e-5)
print("Local attention correctness check: passed")
del tiny_dense, tiny_local, dense_logits, local_logits
gc.collect()


## Обучение и validation

Каждая модель получает одинаковую последовательность мини-батчей и одинаковую инициализацию при одном `seed`. Оптимизатор AdamW применяет weight decay только к матрицам. Learning rate линейно растет во время warmup, затем уменьшается по cosine schedule. Validation loss считается без обновления параметров, perplexity равна экспоненте среднего loss.


In [ ]:
def autocast_context():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return nullcontext()


def make_optimizer(model):
    decay, no_decay = [], []
    for _, parameter in model.named_parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    return torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": cfg.weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=cfg.learning_rate,
        betas=cfg.betas,
        fused=DEVICE.type == "cuda",
    )


def make_scheduler(optimizer):
    warmup_steps = max(1, round(cfg.train_steps * cfg.warmup_ratio))

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, cfg.train_steps - warmup_steps)
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
        return cfg.min_lr_ratio + (1.0 - cfg.min_lr_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.inference_mode()
def evaluate(model, loader, max_batches):
    model.eval()
    losses = []
    for batch_index, (x, y) in enumerate(loader):
        if batch_index >= max_batches:
            break
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with autocast_context():
            loss = model(x, y)["loss"]
        losses.append(float(loss))
    mean_loss = float(np.mean(losses))
    return {"validation_loss": mean_loss, "perplexity": float(math.exp(min(20.0, mean_loss)))}


def infinite_batches(loader):
    while True:
        yield from loader


def train_one(attention_type: str, seed: int = SEED):
    checkpoint_path = PROJECT_DIR / "checkpoints" / f"{attention_type}_{RUN_MODE}_seed{seed}.pt"
    summary_path = PROJECT_DIR / "results" / f"summary_{attention_type}_{RUN_MODE}_seed{seed}.json"
    log_path = PROJECT_DIR / "results" / f"training_{attention_type}_{RUN_MODE}_seed{seed}.csv"
    if checkpoint_path.exists() and summary_path.exists() and not RETRAIN_EXISTING:
        print("Using existing checkpoint:", checkpoint_path)
        return json.loads(summary_path.read_text(encoding="utf-8"))

    seed_everything(seed)
    train_loader, valid_loader = make_loaders(seed)
    stream = infinite_batches(train_loader)
    model = SmallGPT(make_gpt_config(attention_type)).to(DEVICE)
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    log_rows = []
    model.train()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    for step in range(1, cfg.train_steps + 1):
        optimizer.zero_grad(set_to_none=True)
        accumulated_loss = 0.0
        for _ in range(cfg.grad_accum_steps):
            x, y = next(stream)
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast_context():
                loss = model(x, y)["loss"] / cfg.grad_accum_steps
            scaler.scale(loss).backward()
            accumulated_loss += float(loss.detach())
        scaler.unscale_(optimizer)
        grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip))
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        if step == 1 or step % cfg.log_every == 0 or step == cfg.train_steps:
            elapsed = time.perf_counter() - start_time
            tokens_seen = step * TOKENS_PER_OPTIMIZER_STEP
            row = {
                "step": step,
                "train_loss": accumulated_loss,
                "learning_rate": scheduler.get_last_lr()[0],
                "grad_norm": grad_norm,
                "elapsed_seconds": elapsed,
                "train_tokens_per_second": tokens_seen / elapsed,
            }
            log_rows.append(row)
            print(attention_type, row)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    training_seconds = time.perf_counter() - start_time
    validation = evaluate(model, valid_loader, cfg.eval_batches)
    peak_vram = torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else None
    checkpoint = {
        "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "model_config": asdict(model.config),
        "run_config": asdict(cfg),
        "seed": seed,
        "attention_type": attention_type,
    }
    torch.save(checkpoint, checkpoint_path)
    pd.DataFrame(log_rows).to_csv(log_path, index=False)
    summary = {
        "attention_type": attention_type,
        "run_mode": RUN_MODE,
        "seed": seed,
        "parameters": count_parameters(model),
        "model_memory_fp32_bytes": count_parameters(model) * 4,
        "model_memory_fp16_bytes": count_parameters(model) * 2,
        "training_seconds": training_seconds,
        "training_tokens": cfg.train_steps * TOKENS_PER_OPTIMIZER_STEP,
        "training_tokens_per_second": cfg.train_steps * TOKENS_PER_OPTIMIZER_STEP / training_seconds,
        "peak_training_vram_bytes": peak_vram,
        **validation,
        "checkpoint": str(checkpoint_path),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    del model, optimizer, scheduler, scaler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return summary


In [ ]:
# Baseline. После завершения чекпоинт сохраняется, GPU-память освобождается.
baseline_summary = train_one("dense")
baseline_summary


In [ ]:
# Proposed method. Использует тот же seed и тот же протокол обучения.
local_summary = train_one("local")
local_summary


In [ ]:
quality_table = pd.DataFrame([baseline_summary, local_summary])
quality_table["peak_training_vram_gib"] = quality_table["peak_training_vram_bytes"] / 2**30
quality_table["model_memory_fp32_mib"] = quality_table["model_memory_fp32_bytes"] / 2**20
quality_table["model_memory_fp16_mib"] = quality_table["model_memory_fp16_bytes"] / 2**20
quality_table["training_minutes"] = quality_table["training_seconds"] / 60
quality_table.to_csv(PROJECT_DIR / "results" / f"quality_{RUN_MODE}.csv", index=False)
display(quality_table[[
    "attention_type", "parameters", "validation_loss", "perplexity",
    "model_memory_fp32_mib", "model_memory_fp16_mib", "training_minutes",
    "training_tokens_per_second", "peak_training_vram_gib"
]])

ppl_ratio = local_summary["perplexity"] / baseline_summary["perplexity"]
print("Local / dense perplexity ratio:", ppl_ratio)
print("Comparable by the preregistered 5% rule:", ppl_ratio <= 1.05)


## Zero shot LAMBADA

Для каждого примера модель получает контекст без последнего слова. Последнее слово может состоять из нескольких subword-токенов. Метрика считается правильной только тогда, когда argmax модели совпал со всеми токенами продолжения. Предыдущие токены правильного ответа подаются teacher-forced, как в стандартной проверке greedy continuation.


In [ ]:
lambada = load_dataset("EleutherAI/lambada_openai", "default", split="test")

def load_model_from_checkpoint(path: str | Path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = SmallGPT(GPTConfig(**checkpoint["model_config"]))
    model.load_state_dict(checkpoint["model_state"])
    model.eval().to(DEVICE)
    if DEVICE.type == "cuda":
        model.half()
    return model


@torch.inference_mode()
def lambada_exact_match(model, max_examples: int):
    correctness = []
    token_correctness = []
    used = 0
    for example in lambada:
        text = example["text"].strip()
        if " " not in text:
            continue
        context_text, target_word = text.rsplit(" ", 1)
        context_ids = sp.encode(context_text, out_type=int)
        target_ids = sp.encode(" " + target_word, out_type=int)
        if not target_ids:
            continue
        max_context = cfg.seq_len - len(target_ids)
        context_ids = context_ids[-max(1, max_context):]
        if not context_ids:
            context_ids = [sp.bos_id()]
        input_ids = context_ids + target_ids[:-1]
        x = torch.tensor(input_ids, dtype=torch.long, device=DEVICE)[None, :]
        with autocast_context():
            logits = model(x)["logits"]
        start = len(context_ids) - 1
        predictions = logits[0, start : start + len(target_ids)].argmax(dim=-1).cpu().tolist()
        matches = [int(p == y) for p, y in zip(predictions, target_ids)]
        correctness.append(int(all(matches) and len(matches) == len(target_ids)))
        token_correctness.extend(matches)
        used += 1
        if used >= max_examples:
            break
    return {
        "examples": used,
        "exact_match": float(np.mean(correctness)),
        "target_token_accuracy": float(np.mean(token_correctness)),
        "per_example_correct": correctness,
    }


zero_shot_rows = []
zero_shot_vectors = {}
for summary in (baseline_summary, local_summary):
    model = load_model_from_checkpoint(summary["checkpoint"])
    result = lambada_exact_match(model, cfg.lambada_examples)
    zero_shot_vectors[summary["attention_type"]] = result.pop("per_example_correct")
    zero_shot_rows.append({"attention_type": summary["attention_type"], **result})
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

zero_shot_table = pd.DataFrame(zero_shot_rows)
zero_shot_table.to_csv(PROJECT_DIR / "results" / f"lambada_{RUN_MODE}.csv", index=False)
display(zero_shot_table)

def paired_bootstrap_difference(a, b, repeats=5_000, seed=SEED):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(len(a), len(b))
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, n, size=(repeats, n))
    differences = b[:n][indices].mean(axis=1) - a[:n][indices].mean(axis=1)
    return np.quantile(differences, [0.025, 0.5, 0.975]).tolist()

ci = paired_bootstrap_difference(zero_shot_vectors["dense"], zero_shot_vectors["local"])
print("Paired bootstrap 95% interval for local - dense exact match:", ci)


## FLOPs память latency throughput и scaling

Теоретическая оценка считает multiply-add как два FLOPs. Она разделяет общие для моделей проекции и SwiGLU от различающейся стоимости attention. Реальный wall-clock результат может не следовать FLOPs буквально: baseline использует оптимизированный SDPA kernel, а локальный учебный kernel написан на стандартных PyTorch-операциях. Поэтому в выводах нужно показывать и теорию, и фактические измерения на одной GPU.


In [ ]:
def theoretical_flops(attention_type: str, seq_len: int, batch_size: int = 1):
    d, d_ff, layers, vocab = cfg.d_model, cfg.d_ff, cfg.n_layers, cfg.vocab_size
    window = min(cfg.window_size, seq_len)
    projections = batch_size * layers * 8 * seq_len * d * d
    swiglu = batch_size * layers * 6 * seq_len * d * d_ff
    attention = batch_size * layers * 4 * seq_len * (seq_len if attention_type == "dense" else window) * d
    lm_head = batch_size * 2 * seq_len * d * vocab
    total = projections + swiglu + attention + lm_head
    return {
        "projection_flops": projections,
        "swiglu_flops": swiglu,
        "attention_flops": attention,
        "lm_head_flops": lm_head,
        "total_forward_flops": total,
    }


def tensor_bytes(tensor):
    return tensor.numel() * tensor.element_size()


def cache_bytes(past_key_values):
    return sum(tensor_bytes(k) + tensor_bytes(v) for k, v in past_key_values)


@torch.inference_mode()
def benchmark_forward(model, lengths=(128, 256, 512, 1024, 2048), repeats=5):
    rows = []
    for length in lengths:
        x = torch.randint(0, cfg.vocab_size, (1, length), device=DEVICE)
        try:
            for _ in range(2):
                with autocast_context():
                    model(x)
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
            start = time.perf_counter()
            for _ in range(repeats):
                with autocast_context():
                    model(x)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            elapsed = (time.perf_counter() - start) / repeats
            rows.append({
                "context_length": length,
                "latency_ms": elapsed * 1000,
                "throughput_tokens_per_second": length / elapsed,
                "peak_inference_vram_bytes": torch.cuda.max_memory_allocated() if DEVICE.type == "cuda" else None,
            })
        except torch.cuda.OutOfMemoryError:
            rows.append({"context_length": length, "latency_ms": np.nan,
                         "throughput_tokens_per_second": np.nan,
                         "peak_inference_vram_bytes": np.nan})
            torch.cuda.empty_cache()
    return rows


@torch.inference_mode()
def benchmark_cached_decode(model, prompt_length=256, new_tokens=64):
    prompt = torch.randint(0, cfg.vocab_size, (1, prompt_length), device=DEVICE)
    with autocast_context():
        out = model(prompt, use_cache=True, position_offset=0)
    past = out["past_key_values"]
    actual_cache_bytes = cache_bytes(past)
    token = out["logits"][:, -1:].argmax(dim=-1)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for step in range(new_tokens):
        with autocast_context():
            out = model(
                token, past_key_values=past, use_cache=True,
                position_offset=prompt_length + step,
            )
        past = out["past_key_values"]
        token = out["logits"][:, -1:].argmax(dim=-1)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        "prompt_length": prompt_length,
        "generated_tokens": new_tokens,
        "decode_tokens_per_second": new_tokens / elapsed,
        "kv_cache_bytes_after_prefill": actual_cache_bytes,
    }


benchmark_rows = []
decode_rows = []
lengths = (128, 256, 512, 1024, 2048)
for summary in (baseline_summary, local_summary):
    attention_type = summary["attention_type"]
    model = load_model_from_checkpoint(summary["checkpoint"])
    for row in benchmark_forward(model, lengths=lengths):
        flops = theoretical_flops(attention_type, row["context_length"])
        benchmark_rows.append({"attention_type": attention_type, **row, **flops})
    decode_rows.append({"attention_type": attention_type, **benchmark_cached_decode(model)})
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

benchmark_table = pd.DataFrame(benchmark_rows)
decode_table = pd.DataFrame(decode_rows)
benchmark_table.to_csv(PROJECT_DIR / "results" / f"scaling_{RUN_MODE}.csv", index=False)
decode_table.to_csv(PROJECT_DIR / "results" / f"decode_{RUN_MODE}.csv", index=False)
display(benchmark_table)
display(decode_table)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for attention_type, group in benchmark_table.groupby("attention_type"):
    axes[0].plot(group["context_length"], group["latency_ms"], marker="o", label=attention_type)
    axes[1].plot(group["context_length"], group["peak_inference_vram_bytes"] / 2**30, marker="o", label=attention_type)
    axes[2].plot(group["context_length"], group["total_forward_flops"] / 1e9, marker="o", label=attention_type)
axes[0].set(title="Forward latency", xlabel="Context length", ylabel="Milliseconds")
axes[1].set(title="Peak inference VRAM", xlabel="Context length", ylabel="GiB")
axes[2].set(title="Approximate forward FLOPs", xlabel="Context length", ylabel="GFLOPs")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
figure_path = PROJECT_DIR / "figures" / f"scaling_{RUN_MODE}.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved:", figure_path)


In [ ]:
def gib(value):
    return None if value is None else value / 2**30

final_report = {
    "environment": {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    },
    "run_config": asdict(cfg),
    "baseline": baseline_summary,
    "proposed": local_summary,
    "perplexity_ratio_local_over_dense": ppl_ratio,
    "quality_comparable_at_5_percent": bool(ppl_ratio <= 1.05),
    "lambada": zero_shot_rows,
    "lambada_paired_bootstrap_local_minus_dense": ci,
    "decode": decode_rows,
}
report_path = PROJECT_DIR / "results" / f"final_report_{RUN_MODE}_seed{SEED}.json"
report_path.write_text(json.dumps(final_report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Final report:", report_path)


## Как сделать научный вывод

1. Сначала убедитесь, что `smoke` завершается без ошибок.
2. Проведите `quick` и проверьте, что loss обеих моделей снижается.
3. Удалите или переименуйте старые чекпоинты, выберите `full` и выполните обе тренировки на одной модели GPU.
4. Для итоговой работы повторите full-эксперимент с seed 42, 43 и 44, если хватает вычислительного бюджета. Сведите среднее и стандартное отклонение.
5. Качество считается сопоставимым по заранее заданному правилу, если `PPL_local / PPL_dense <= 1.05`. LAMBADA трактуется как дополнительная метрика, особенно если число правильных ответов у модели мало.
6. Не утверждайте ускорение только по FLOPs. Покажите latency и throughput. Учебная local-attention реализация может проигрывать оптимизированному dense SDPA kernel на коротком контексте, даже если выполняет меньше арифметических операций.

Основные файлы для отчета находятся в папке `results`, графики в `figures`, а веса в `checkpoints`.
